In [ ]:
%%bash
# Clone Kohya sd-scripts for state-of-the-art latent diffusion fine-tuning
git clone https://github.com/kohya-ss/sd-scripts.git /kaggle/working/sd-scripts
cd /kaggle/working/sd-scripts

# Upgrade pip and install core PyTorch/HuggingFace dependencies
pip install --upgrade pip
pip install -r requirements.txt

# Inject advanced optimizers, memory-efficient attention, and mixed-precision handlers
pip install xformers bitsandbytes accelerate prodigyopt

# Initialize default accelerate configuration for single-node Kaggle GPU topography
accelerate config default

In [ ]:
import os
import glob
from PIL import Image

# Define I/O pipelines mapping to Kohya's expected folder hierarchy (40 repeats for captionless)
input_dir = "/kaggle/input/datasets/anindoakb/guinevere-lora"
output_dir = "/kaggle/working/train_guinevere/img/40_guinevere_mlbb"
os.makedirs(output_dir, exist_ok=True)

image_paths = glob.glob(os.path.join(input_dir, "*.*"))

# Ingest, enforce RGB tensor normalization, and synthesize contralateral views 
for idx, img_path in enumerate(image_paths):
    try:
        img = Image.open(img_path).convert("RGB")
        
        # Base image: Write as high-quality JPG to prevent uncompressed tensor bottlenecks
        base_out = os.path.join(output_dir, f"img_{idx:04d}.jpg")
        img.save(base_out, "JPEG", quality=100)
        
        # Affine horizontal flip: Artificially populates the latent space with 360-degree symmetries
        img_flip = img.transpose(Image.FLIP_LEFT_RIGHT)
        flip_out = os.path.join(output_dir, f"img_{idx:04d}_aug_flip.jpg")
        img_flip.save(flip_out, "JPEG", quality=100)
        
    except Exception as e:
        print(f"Skipping {img_path} due to tensor decode exception: {e}")

print(f"Synthesis complete. Generated {len(os.listdir(output_dir))} spatial samples.")

In [ ]:
%%bash
cd /kaggle/working/sd-scripts
mkdir -p /kaggle/working/output_guinevere

# Launch multi-threaded precision training
# - Prodigy dynamically calculates d_coef based learning rates scaling
# - Min-SNR gamma smooths loss variance on noisy timesteps
# - Multires noise mitigates CFG burn and pushes model to generate structural detail
accelerate launch --num_cpu_threads_per_process=2 train_network.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_guinevere/img" \
  --output_dir="/kaggle/working/output_guinevere" \
  --output_name="guinevere_lora" \
  --network_module="networks.lora" \
  --network_dim=256 \
  --network_alpha=128 \
  --optimizer_type="Prodigy" \
  --learning_rate=1.0 \
  --unet_lr=1.0 \
  --text_encoder_lr=1.0 \
  --lr_scheduler="cosine" \
  --optimizer_args weight_decay=0.01 d_coef=2 use_bias_correction=True safeguard_warmup=True \
  --min_snr_gamma=5.0 \
  --noise_offset=0.1 \
  --multires_noise_iterations=6 \
  --multires_noise_discount=0.3 \
  --network_dropout=0.1 \
  --scale_weight_norms=1.0 \
  --color_aug \
  --random_crop \
  --mixed_precision="bf16" \
  --save_precision="bf16" \
  --xformers \
  --gradient_checkpointing \
  --max_train_epochs=30 \
  --save_every_n_epochs=5 \
  --train_batch_size=2 \
  --resolution="512,512" \
  --save_model_as="safetensors" \
  --cache_latents

In [ ]:
import shutil
import os

# Compile the output weights into a single contiguous archive for Kaggle download
output_folder = "/kaggle/working/output_guinevere"
archive_name = "/kaggle/working/guinevere_lora_outputs"

if os.path.exists(output_folder):
    shutil.make_archive(archive_name, 'zip', output_folder)
    print(f"Weights serialized and compressed to {archive_name}.zip")
else:
    print("Fatal: Output directory not found. Did the training script fault?")